# SQL interview drills

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-10-interview-drills.ipynb)

Twenty-two coding problems and fourteen concept questions, all against one small company dataset. Each problem
states the result it should return. Each answer runs the query, shows the rows, and asserts they match.

**Try each problem before you scroll down.** Write the query, run it, then compare it with the answer and read
the notes. Use the [basics](sql-01-basics.ipynb) notebook for the building blocks.

Spark SQL differs from the original DuckDB question bank in a few places, so the drills use these workarounds:

- Temporary views are read-only, so `DELETE` and `UPDATE` become "keep the survivors" and "apply the latest
  update with a fallback". The expected results are unchanged.
- Spark tables have no `rowid`, so the update history carries an explicit `update_id`.
- Spark has no `QUALIFY` clause and no `DISTINCT` inside a window. The filtered and cumulative variants use a
  subquery and a first-order running sum.

## Setup

Install PySpark, start a Spark session, and define a small helper that returns query results as Python tuples.

In [ ]:
!pip install -q pyspark

In [ ]:
from datetime import date, datetime
from decimal import Decimal

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


def rows(query):
    """Run a query and return its rows as tuples; DECIMAL values become floats."""
    return [tuple(float(v) if isinstance(v, Decimal) else v for v in row) for row in spark.sql(query).collect()]

## The drill data

The six tables are the same data as the original question bank, as Spark temporary views. A
`CREATE TEMP VIEW ... AS SELECT * FROM VALUES` cell creates each one.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW departments AS
SELECT * FROM VALUES
  (1, 'Engineering', 'New York'),
  (2, 'Sales', 'Chicago'),
  (3, 'Marketing', 'Austin'),
  (4, 'Support', 'Remote')
AS departments(dept_id, dept_name, location)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW employees AS
SELECT * FROM VALUES
  (1, 'Alice', 1, NULL, 150000, DATE '2018-01-15', 'alice@example.com'),
  (6, 'Frank', 2, NULL, 130000, DATE '2017-05-20', 'frank@example.com'),
  (10, 'Judy', 3, NULL, 105000, DATE '2019-11-01', 'judy@example.com'),
  (12, 'Leo', 4, NULL, 90000, DATE '2018-08-08', 'leo@example.com'),
  (7, 'Grace', 2, 6, 98000, DATE '2020-02-14', 'grace@example.com'),
  (8, 'Heidi', 2, 6, 87000, DATE '2021-07-01', 'heidi@example.com'),
  (9, 'Ivan', 2, 6, 87000, DATE '2022-03-15', 'ivan@example.com'),
  (11, 'Kevin', 3, 10, 88000, DATE '2021-04-01', 'dave@example.com'),
  (2, 'Bob', 1, 1, 120000, DATE '2019-03-01', 'bob@example.com'),
  (3, 'Carol', 1, 1, 120000, DATE '2020-06-15', 'carol@example.com'),
  (13, 'Mallory', 4, 12, 72000, DATE '2022-05-05', 'mallory@example.com'),
  (14, 'Nina', 4, 12, 68000, DATE '2023-02-01', 'heidi@example.com'),
  (4, 'Dave', 1, 2, 95000, DATE '2021-09-01', 'dave@example.com'),
  (5, 'Eve', 1, 2, 90000, DATE '2022-01-10', 'eve@example.com')
AS employees(emp_id, name, dept_id, manager_id, salary, hire_date, email)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW employee_updates AS
SELECT * FROM VALUES
  (1, 1, 1, 155000, DATE '2023-01-01'),
  (2, 1, 1, 165000, DATE '2023-06-01'),
  (3, 2, 1, 122000, DATE '2023-02-01'),
  (4, 2, 1, 128000, DATE '2023-05-01'),
  (5, 3, 1, 125000, DATE '2023-03-01'),
  (6, 6, 2, 135000, DATE '2023-01-15'),
  (7, 6, 2, 140000, DATE '2023-07-01'),
  (8, 7, 2, 101000, DATE '2023-04-01')
AS employee_updates(update_id, emp_id, dept_id, salary, updated_at)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp', 'North', DATE '2022-01-05'),
  (2, 'Beta LLC', 'North', DATE '2022-02-10'),
  (3, 'Cyan Inc', 'South', DATE '2022-03-15'),
  (4, 'Delta Co', 'South', DATE '2022-04-20'),
  (5, 'Echo Ltd', 'West', DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West', DATE '2022-06-30'),
  (7, 'Gamma SA', 'East', DATE '2023-01-10'),
  (8, 'Hotel BV', 'East', DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1, 1, 'completed', 100, DATE '2023-01-05'),
  (2, 1, 'completed', 200, DATE '2023-02-10'),
  (3, 1, 'completed', 150, DATE '2023-03-15'),
  (4, 2, 'completed', 300, DATE '2023-01-20'),
  (5, 2, 'cancelled', 250, DATE '2023-02-05'),
  (6, 2, 'completed', 400, DATE '2023-03-20'),
  (7, 3, 'completed', 120, DATE '2023-01-25'),
  (8, 3, 'completed', 180, DATE '2023-02-28'),
  (9, 4, 'pending', 220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW events AS
SELECT * FROM VALUES
  (1, 1, TIMESTAMP '2023-01-01 09:00:00', 'login'),
  (2, 1, TIMESTAMP '2023-01-02 09:05:00', 'login'),
  (3, 1, TIMESTAMP '2023-01-03 08:55:00', 'login'),
  (4, 1, TIMESTAMP '2023-01-05 09:00:00', 'login'),
  (5, 1, TIMESTAMP '2023-01-06 09:10:00', 'login'),
  (6, 2, TIMESTAMP '2023-01-01 10:00:00', 'login'),
  (7, 2, TIMESTAMP '2023-01-02 10:00:00', 'login'),
  (8, 2, TIMESTAMP '2023-01-04 10:00:00', 'login'),
  (9, 3, TIMESTAMP '2023-02-01 10:00:00', 'view'),
  (10, 3, TIMESTAMP '2023-02-01 10:05:00', 'click'),
  (11, 3, TIMESTAMP '2023-02-01 10:25:00', 'view'),
  (12, 3, TIMESTAMP '2023-02-01 11:30:00', 'purchase'),
  (13, 3, TIMESTAMP '2023-02-01 11:45:00', 'view'),
  (14, 4, TIMESTAMP '2023-02-01 10:00:00', 'login'),
  (15, 4, TIMESTAMP '2023-02-01 10:20:00', 'view')
AS events(event_id, user_id, event_ts, event_type)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tiny_orders AS
SELECT * FROM VALUES
  (1),
  (NULL)
AS tiny_orders(customer_id)
""")

In [ ]:
q = """
    SELECT
        (SELECT COUNT(*) FROM departments)      AS departments,
        (SELECT COUNT(*) FROM employees)        AS employees,
        (SELECT COUNT(*) FROM employee_updates) AS employee_updates,
        (SELECT COUNT(*) FROM customers)        AS customers,
        (SELECT COUNT(*) FROM orders)           AS orders,
        (SELECT COUNT(*) FROM events)           AS events
    """
spark.sql(q).show()
assert rows(q) == [(4, 14, 8, 8, 14, 15)]

## Part A - Ranking and top-N

`ROW_NUMBER`, `RANK` and `DENSE_RANK` all number rows within a partition, and they break ties differently. Most
"top N per group" questions are really questions about which one you need.

### Q1. Second-highest salary

Return the second-highest salary in `employees`. Repeated salaries count as one value.

**Expected result:** one row, `(130000,)`.

In [ ]:
q = """
    SELECT MAX(salary) AS second_highest
    FROM employees
    WHERE salary < (SELECT MAX(salary) FROM employees)
    """
spark.sql(q).show()
assert rows(q) == [(130000,)]
q = """
    SELECT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
    )
    WHERE rnk = 2
    """
spark.sql(q).show()
assert rows(q) == [(130000,)]

**Approach.** `MAX` below the overall maximum is the runner-up, because `<` excludes every copy of the top salary.
The trap is `ORDER BY salary DESC LIMIT 1 OFFSET 1`, which returns the top salary again when it is tied.

### Q2. Nth-highest salary

Return the third-highest distinct salary. The distinct salaries in descending order start 150000, 130000, 120000.

**Expected result:** one row, `(120000,)`.

In [ ]:
q = "SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 1 OFFSET 2"
spark.sql(q).show()
assert rows(q) == [(120000,)]
q = """
    SELECT DISTINCT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
    )
    WHERE rnk = 3
    """
spark.sql(q).show()
assert rows(q) == [(120000,)]
q = """
    SELECT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
    )
    WHERE rnk = 3
    ORDER BY salary DESC
    """
spark.sql(q).show()
assert rows(q) == [(120000,), (120000,)]

**Approach.** `DENSE_RANK` gives one rank per distinct value, so a tie produces two rows at the same rank. `DISTINCT`
removes the duplicate. `ROW_NUMBER` never emits ties, but it has no stable definition of "Nth" when values repeat.

### Q3. Top-3 per department, and how each ranking function counts

Return the top three employees by salary in each department. Then compare how many rows each ranking function keeps.

**Expected result:** eleven rows. Department 2 has three rows because Heidi and Ivan tie at 87000 and `ROW_NUMBER`
breaks the tie with `emp_id`.

In [ ]:
q = """
    SELECT dept_id, name, salary FROM (
        SELECT dept_id, name, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    )
    WHERE rn <= 3
    ORDER BY dept_id, salary DESC, name
    """
spark.sql(q).show()
assert rows(q) == [
        (1, "Alice", 150000), (1, "Bob", 120000), (1, "Carol", 120000),
        (2, "Frank", 130000), (2, "Grace", 98000), (2, "Heidi", 87000),
        (3, "Judy", 105000), (3, "Kevin", 88000),
        (4, "Leo", 90000), (4, "Mallory", 72000), (4, "Nina", 68000),
    ]
q = """
    SELECT dept_id,
           COUNT(CASE WHEN rn   <= 3 THEN 1 END) AS by_row_number,
           COUNT(CASE WHEN rnk  <= 3 THEN 1 END) AS by_rank,
           COUNT(CASE WHEN drnk <= 3 THEN 1 END) AS by_dense_rank
    FROM (
        SELECT dept_id, salary, emp_id,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn,
               RANK()       OVER (PARTITION BY dept_id ORDER BY salary DESC)         AS rnk,
               DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC)         AS drnk
        FROM employees
    )
    GROUP BY dept_id
    ORDER BY dept_id
    """
spark.sql(q).show()
assert rows(q) == [(1, 3, 3, 4), (2, 3, 4, 4), (3, 2, 2, 2), (4, 3, 3, 3)]
q = """
    SELECT name, salary,
           RANK()       OVER by_salary AS rnk,
           DENSE_RANK() OVER by_salary AS drnk
    FROM employees
    WHERE dept_id = 1
    WINDOW by_salary AS (ORDER BY salary DESC)
    ORDER BY salary DESC, emp_id
    """
spark.sql(q).show()
assert rows(q) == [
        ("Alice", 150000, 1, 1), ("Bob", 120000, 2, 2), ("Carol", 120000, 2, 2),
        ("Dave", 95000, 4, 3), ("Eve", 90000, 5, 4),
    ]

**Approach.** Filter on a ranking in an outer query, because a window cannot appear in `WHERE`. Use `ROW_NUMBER` for
"exactly N per group", and `RANK` or `DENSE_RANK` when all tied rows should survive. Always add a unique tie-breaker
to `ROW_NUMBER`'s `ORDER BY`.

## Part B - Deduplication and history

Real tables arrive with duplicates, from append-only history, retried loads and dirty sources. These questions
collapse many rows per key into one, safely and reproducibly.

### Q4. Keep the latest row per employee

`employee_updates` has several rows per `emp_id`. Return the most recent row for each employee.

**Expected result:** five rows, one per employee updated.

In [ ]:
q = """
    SELECT emp_id, dept_id, salary, updated_at FROM (
        SELECT emp_id, dept_id, salary, updated_at,
               ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, update_id DESC) AS rn
        FROM employee_updates
    )
    WHERE rn = 1
    ORDER BY emp_id
    """
spark.sql(q).show()
assert rows(q) == [
        (1, 1, 165000, date(2023, 6, 1)),
        (2, 1, 128000, date(2023, 5, 1)),
        (3, 1, 125000, date(2023, 3, 1)),
        (6, 2, 140000, date(2023, 7, 1)),
        (7, 2, 101000, date(2023, 4, 1)),
    ]
q = """
    SELECT u.emp_id, u.dept_id, u.salary, u.updated_at
    FROM employee_updates u
    JOIN (
        SELECT emp_id, MAX(updated_at) AS max_ts FROM employee_updates GROUP BY emp_id
    ) m ON m.emp_id = u.emp_id AND m.max_ts = u.updated_at
    ORDER BY u.emp_id
    """
spark.sql(q).show()
assert rows(q) == [
        (1, 1, 165000, date(2023, 6, 1)),
        (2, 1, 128000, date(2023, 5, 1)),
        (3, 1, 125000, date(2023, 3, 1)),
        (6, 2, 140000, date(2023, 7, 1)),
        (7, 2, 101000, date(2023, 4, 1)),
    ]

**Approach.** `ROW_NUMBER` ordered by the timestamp, with a unique tie-breaker, is the general recipe. The
`GROUP BY MAX` join is only correct when the maximum is unique. Spark's `DISTINCT` and `QUALIFY` alternatives are
covered in the window-functions notebook.

### Q5. Delete duplicates, keeping one row per employee

Spark temporary views cannot be modified, so express the delete as the set of rows to keep.

**Expected result:** five survivors, the latest update per employee.

In [ ]:
q = """
    SELECT u.update_id, u.emp_id, u.salary
    FROM employee_updates u
    WHERE NOT EXISTS (
        SELECT 1 FROM employee_updates k
        WHERE k.emp_id = u.emp_id
          AND (k.updated_at > u.updated_at
               OR (k.updated_at = u.updated_at AND k.update_id > u.update_id))
    )
    ORDER BY u.emp_id
    """
spark.sql(q).show()
assert rows(q) == [(2, 1, 165000), (4, 2, 128000), (5, 3, 125000), (7, 6, 140000), (8, 7, 101000)]

**Approach.** "Identify survivors, delete everything else" is the safe pattern. On a Delta or Iceberg table you would
run `DELETE FROM ... WHERE update_id NOT IN (survivors)` or a `MERGE`; the predicate is the same.

### Q6. Find duplicate emails

List every email used by more than one employee, with the names sharing it.

**Expected result:** two rows, `dave@example.com` (Dave, Kevin) and `heidi@example.com` (Heidi, Nina).

In [ ]:
q = """
    SELECT email, COUNT(*) AS copies, array_join(sort_array(collect_list(name)), ',') AS people
    FROM employees
    GROUP BY email
    HAVING COUNT(*) > 1
    ORDER BY email
    """
spark.sql(q).show()
assert rows(q) == [("dave@example.com", 2, "Dave,Kevin"), ("heidi@example.com", 2, "Heidi,Nina")]
q = """
    SELECT emp_id, email FROM employees
    WHERE email IN (SELECT email FROM employees GROUP BY email HAVING COUNT(*) > 1)
    ORDER BY email, emp_id
    """
spark.sql(q).show()
assert rows(q) == [(4, "dave@example.com"), (11, "dave@example.com"), (8, "heidi@example.com"), (14, "heidi@example.com")]

**Approach.** `GROUP BY email HAVING COUNT(*) > 1` is the whole answer. Real pipelines normalise the key first with
`lower(trim(email))`. `array_join(sort_array(collect_list(...)))` gives a deterministic list of names.

### Q7. Apply the latest salary update, leaving other employees alone

Use the newest update per employee and keep the current salary for employees with no update.

**Expected result:** five salaries change (Alice, Bob, Carol, Frank and Grace); the other nine keep their values.

In [ ]:
CURRENT_SALARY_SQL = """
    WITH latest AS (
        SELECT emp_id, salary FROM (
            SELECT emp_id, salary,
                   ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, update_id DESC) AS rn
            FROM employee_updates
        )
        WHERE rn = 1
    )
    SELECT e.emp_id, COALESCE(l.salary, e.salary) AS salary
    FROM employees e
    LEFT JOIN latest l ON l.emp_id = e.emp_id
"""
q = CURRENT_SALARY_SQL + " ORDER BY emp_id"
spark.sql(q).show()
assert rows(q) == [
        (1, 165000), (2, 128000), (3, 125000), (4, 95000), (5, 90000),
        (6, 140000), (7, 101000), (8, 87000), (9, 87000), (10, 105000),
        (11, 88000), (12, 90000), (13, 72000), (14, 68000),
    ]
q = f"""
    SELECT COUNT(*) AS changed_rows
    FROM ({CURRENT_SALARY_SQL}) n
    JOIN employees o ON o.emp_id = n.emp_id AND o.salary <> n.salary
    """
spark.sql(q).show()
assert rows(q) == [(5,)]

**Approach.** The `COALESCE` fallback is what keeps employees without an update from becoming `NULL`. A real table
would take the same result through `MERGE`, after the source has been deduplicated to one row per key.

## Part C - Running totals and period-over-period growth

Window aggregates keep every row while adding a cumulative or comparative value. Order the window by a unique key
when the ordering column has ties.

### Q8. Running total across all orders

For every order in date order, show the cumulative amount across all orders.

**Expected result:** fourteen rows, ending at 3790.

In [ ]:
q = """
    SELECT order_id, order_date, amount,
           SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
    FROM orders
    ORDER BY order_date, order_id
    """
spark.sql(q).show()
assert rows(q) == [
        (11, date(2023, 1, 2), 500, 500), (1, date(2023, 1, 5), 100, 600),
        (4, date(2023, 1, 20), 300, 900), (7, date(2023, 1, 25), 120, 1020),
        (5, date(2023, 2, 5), 250, 1270), (2, date(2023, 2, 10), 200, 1470),
        (9, date(2023, 2, 14), 220, 1690), (13, date(2023, 2, 18), 275, 1965),
        (8, date(2023, 2, 28), 180, 2145), (12, date(2023, 3, 5), 450, 2595),
        (3, date(2023, 3, 15), 150, 2745), (6, date(2023, 3, 20), 400, 3145),
        (14, date(2023, 3, 25), 325, 3470), (10, date(2023, 3, 30), 320, 3790),
    ]

**Approach.** The default frame for an ordered `SUM` is `RANGE`, so rows that share an `ORDER BY` value share one
total. Adding the unique `order_id` makes the total advance row by row.

### Q9. Running total per customer, completed orders only

**Expected result:** eleven rows. Customer 1 runs 100, 300, 450; customer 5 runs 500 then 950.

In [ ]:
q = """
    SELECT customer_id, order_date, amount,
           SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
    FROM orders
    WHERE status = 'completed'
    ORDER BY customer_id, order_date, order_id
    """
spark.sql(q).show()
assert rows(q) == [
        (1, date(2023, 1, 5), 100, 100), (1, date(2023, 2, 10), 200, 300),
        (1, date(2023, 3, 15), 150, 450), (2, date(2023, 1, 20), 300, 300),
        (2, date(2023, 3, 20), 400, 700), (3, date(2023, 1, 25), 120, 120),
        (3, date(2023, 2, 28), 180, 300), (4, date(2023, 3, 30), 320, 320),
        (5, date(2023, 1, 2), 500, 500), (5, date(2023, 3, 5), 450, 950),
        (6, date(2023, 2, 18), 275, 275),
    ]

**Approach.** `WHERE` runs before the window. Filtering after it would keep the excluded amounts in the running sum.

### Q10. Month-over-month revenue growth

Using completed orders, show monthly revenue, the previous month's revenue and the growth percentage.

**Expected result:** January 1020 (no previous month), February 655 at -35.78%, March 1320 at +101.53%.

In [ ]:
q = """
    WITH monthly AS (
        SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY date_format(order_date, 'yyyy-MM')
    )
    SELECT month, revenue,
           LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 2) AS growth_pct
    FROM monthly
    ORDER BY month
    """
spark.sql(q).show()
assert rows(q) == [("2023-01", 1020, None, None), ("2023-02", 655, 1020, -35.78), ("2023-03", 1320, 655, 101.53)]
q = """
    WITH monthly AS (
        SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY date_format(order_date, 'yyyy-MM')
    )
    SELECT month, ROUND(revenue * 1.0 / LAG(revenue) OVER (ORDER BY month), 3) AS ratio
    FROM monthly
    ORDER BY month
    """
spark.sql(q).show()
assert rows(q) == [("2023-01", None), ("2023-02", 0.642), ("2023-03", 2.015)]

**Approach.** Aggregate to one row per month first, then apply `LAG`. Applied to raw orders, `LAG` returns the previous
order. Keep the first period's growth `NULL` rather than `0`, so "no data" is not confused with "no growth".

### Q11. Each customer's share of completed revenue

**Expected result:** six customers. Customer 5 is the largest at 31.72%, and the shares sum to 100.00.

In [ ]:
q = """
    SELECT customer_id, SUM(amount) AS total,
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    ORDER BY customer_id
    """
spark.sql(q).show()
assert rows(q) == [(1, 450, 15.03), (2, 700, 23.37), (3, 300, 10.02), (4, 320, 10.68), (5, 950, 31.72), (6, 275, 9.18)]
q = """
    SELECT ROUND(SUM(pct), 2) FROM (
        SELECT ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct
        FROM orders WHERE status = 'completed' GROUP BY customer_id
    )
    """
spark.sql(q).show()
assert rows(q) == [(100.0,)]

**Approach.** `SUM(SUM(amount)) OVER ()` nests a window over an aggregate. Multiply by `100.0` before dividing to avoid
integer division. Rounded shares are not guaranteed to total 100 in general, so check it.

## Part D - Joins and set logic

Self-joins, anti-joins and `LEFT JOIN` with an `IS NULL` test are the core of dimensional queries. The question
usually hides a `NULL` or an `EXISTS` versus `IN` decision.

### Q12. Each employee with their manager's name

Employees without a manager must still appear.

**Expected result:** fourteen rows. Alice, Frank, Judy and Leo show `NULL` for the manager.

In [ ]:
q = """
    SELECT e.emp_id, e.name, m.name AS manager
    FROM employees e
    LEFT JOIN employees m ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
    """
spark.sql(q).show()
assert rows(q) == [
        (1, "Alice", None), (2, "Bob", "Alice"), (3, "Carol", "Alice"),
        (4, "Dave", "Bob"), (5, "Eve", "Bob"),
        (6, "Frank", None), (7, "Grace", "Frank"), (8, "Heidi", "Frank"), (9, "Ivan", "Frank"),
        (10, "Judy", None), (11, "Kevin", "Judy"),
        (12, "Leo", None), (13, "Mallory", "Leo"), (14, "Nina", "Leo"),
    ]
q = """
    SELECT m.emp_id, m.name, COUNT(e.emp_id) AS reports
    FROM employees m
    LEFT JOIN employees e ON e.manager_id = m.emp_id
    GROUP BY m.emp_id, m.name
    HAVING COUNT(e.emp_id) = 0
    ORDER BY m.emp_id
    """
spark.sql(q).show()
assert rows(q) == [(3, "Carol", 0), (4, "Dave", 0), (5, "Eve", 0), (7, "Grace", 0), (8, "Heidi", 0),
     (9, "Ivan", 0), (11, "Kevin", 0), (13, "Mallory", 0), (14, "Nina", 0)]

**Approach.** `LEFT JOIN` keeps the rows whose `manager_id` is `NULL`, which an inner join would drop. `COUNT(e.emp_id)`
counts the real matches, not the `NULL` placeholder row. To walk more than one level, use a recursive CTE.

### Q13. Customers with no orders (anti-join)

**Expected result:** `(7, 'Gamma SA')` and `(8, 'Hotel BV')`.

In [ ]:
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [(7, "Gamma SA"), (8, "Hotel BV")]
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [(7, "Gamma SA"), (8, "Hotel BV")]

**Approach.** `NOT EXISTS` is the safest anti-join. `LEFT JOIN ... IS NULL` is equivalent only when the right-hand key is
non-nullable. `NOT IN` is the dangerous one: a single `NULL` in the subquery makes it return no rows.

### Q14. Order counts and totals per customer, including customers with none

**Expected result:** eight rows. Gamma SA and Hotel BV show zero, not `NULL`. `COUNT(*)` would show 1 for them.

In [ ]:
q = """
    SELECT c.customer_id, c.name,
           COUNT(o.order_id) AS order_count,
           COALESCE(SUM(o.amount), 0) AS total_amount
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [
        (1, "Acme Corp", 3, 450), (2, "Beta LLC", 3, 950), (3, "Cyan Inc", 2, 300),
        (4, "Delta Co", 2, 540), (5, "Echo Ltd", 2, 950), (6, "Foxtrot GmbH", 2, 600),
        (7, "Gamma SA", 0, 0), (8, "Hotel BV", 0, 0),
    ]
q = """
    SELECT c.customer_id, COUNT(*) AS rows_seen
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id
    HAVING COUNT(*) = 1
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [(7, 1), (8, 1)]

**Approach.** `COUNT(column)` skips `NULL`, `COUNT(*)` does not. `SUM` over zero rows is `NULL`, so wrap it in
`COALESCE`. A predicate on the right-hand table in `WHERE` quietly turns the outer join into an inner join.

### Q15. `EXISTS` versus `IN`, and the `NOT IN` trap

Find customers with at least one order, first with `EXISTS` and then with `IN`. Then see what a `NULL` does to
`NOT IN`.

**Expected result:** customers 1 through 6 for both forms.

In [ ]:
q = """
    SELECT c.customer_id, c.name FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [(1, "Acme Corp"), (2, "Beta LLC"), (3, "Cyan Inc"), (4, "Delta Co"), (5, "Echo Ltd"), (6, "Foxtrot GmbH")]
q = """
    SELECT customer_id, name FROM customers
    WHERE customer_id IN (SELECT customer_id FROM orders)
    ORDER BY customer_id
    """
spark.sql(q).show()
assert rows(q) == [(1, "Acme Corp"), (2, "Beta LLC"), (3, "Cyan Inc"), (4, "Delta Co"), (5, "Echo Ltd"), (6, "Foxtrot GmbH")]
q = "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)"
spark.sql(q).show()
assert rows(q) == []
q = """
    SELECT c.customer_id FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]

**Approach.** `EXISTS` is a semi-join and is `NULL`-safe. `NOT IN` with a `NULL` in the list returns no rows at all,
because `x <> NULL` is unknown. Readability and `NULL` safety usually decide the choice; optimisers often produce
the same plan.

## Part E - Aggregation, conditional counts and medians

Conditional aggregation turns categories into columns. Medians need an interpolation rule, and the rule matters when the
row count is even.

### Q16. Employees paid above their department average

**Expected result:** six employees. Alice, Bob and Carol in Engineering (average 115000), Frank in Sales (100500),
Judy in Marketing (96500) and Leo in Support (76666.67).

In [ ]:
q = """
    SELECT e.name, e.dept_id, e.salary
    FROM employees e
    WHERE e.salary > (SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id)
    ORDER BY e.dept_id, e.salary DESC, e.name
    """
spark.sql(q).show()
assert rows(q) == [
        ("Alice", 1, 150000), ("Bob", 1, 120000), ("Carol", 1, 120000),
        ("Frank", 2, 130000), ("Judy", 3, 105000), ("Leo", 4, 90000),
    ]
q = """
    SELECT name, dept_id, salary FROM (
        SELECT name, dept_id, salary, AVG(salary) OVER (PARTITION BY dept_id) AS dept_avg
        FROM employees
    )
    WHERE salary > dept_avg
    ORDER BY dept_id, salary DESC, name
    """
spark.sql(q).show()
assert rows(q) == [
        ("Alice", 1, 150000), ("Bob", 1, 120000), ("Carol", 1, 120000),
        ("Frank", 2, 130000), ("Judy", 3, 105000), ("Leo", 4, 90000),
    ]
q = "SELECT dept_id, ROUND(AVG(salary), 2) AS avg_salary FROM employees GROUP BY dept_id ORDER BY dept_id"
spark.sql(q).show()
assert rows(q) == [(1, 115000.0), (2, 100500.0), (3, 96500.0), (4, 76666.67)]

**Approach.** The window version computes each department average once per partition. Use a strict `>`; `>=` would pull in
a department where every salary equals the average.

### Q17. Pivot counts by status

One row per customer, with the number of completed, pending and cancelled orders.

**Expected result:** eight rows. Beta LLC has two completed and one cancelled; the two customers without orders show zeros.

In [ ]:
q = """
    SELECT c.customer_id, c.name,
           COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
           COUNT(CASE WHEN o.status = 'pending'   THEN 1 END) AS pending,
           COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
    """
spark.sql(q).show()
assert rows(q) == [
        (1, "Acme Corp", 3, 0, 0), (2, "Beta LLC", 2, 0, 1), (3, "Cyan Inc", 2, 0, 0),
        (4, "Delta Co", 1, 1, 0), (5, "Echo Ltd", 2, 0, 0), (6, "Foxtrot GmbH", 1, 0, 1),
        (7, "Gamma SA", 0, 0, 0), (8, "Hotel BV", 0, 0, 0),
    ]
q = """
    SELECT customer_id,
           COALESCE(completed, 0) AS completed,
           COALESCE(pending, 0)   AS pending,
           COALESCE(cancelled, 0) AS cancelled
    FROM (
        SELECT * FROM (SELECT customer_id, status FROM orders)
        PIVOT (COUNT(*) FOR status IN ('completed' AS completed, 'pending' AS pending, 'cancelled' AS cancelled))
    )
    ORDER BY customer_id
    """
spark.sql(q).show()
assert rows(q) == [(1, 3, 0, 0), (2, 2, 0, 1), (3, 2, 0, 0), (4, 1, 1, 0), (5, 2, 0, 0), (6, 1, 0, 1)]

**Approach.** `COUNT(CASE ... THEN 1 END)` is the portable form: no `ELSE` gives `NULL`, which `COUNT` ignores. With `SUM`,
you need `ELSE 0`. Spark's `PIVOT` produces `NULL` for missing combinations and drops groups with no rows.

### Q18. Median salary

Compute the median salary in two ways, so you can explain both.

**Expected result:** `92500.0` for both. There are fourteen salaries, so the median is the average of the seventh and
eighth values, 90000 and 95000.

In [ ]:
q = "SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY salary) AS median FROM employees"
spark.sql(q).show()
assert rows(q) == [(92500.0,)]
q = """
    WITH ranked AS (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n FROM employees
    )
    SELECT AVG(salary) FROM ranked
    WHERE rn IN (FLOOR((n + 1) / 2), FLOOR((n + 2) / 2))
    """
spark.sql(q).show()
assert rows(q) == [(92500.0,)]
q = """
    SELECT salary FROM (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n FROM employees
    )
    WHERE rn IN (FLOOR((n + 1) / 2), FLOOR((n + 2) / 2))
    ORDER BY salary
    """
spark.sql(q).show()
assert rows(q) == [(90000,), (95000,)]

**Approach.** `percentile_cont` interpolates between the two middle values, so an even count gives their mean. The window
version does the same by hand and is what you reach for when the engine lacks a percentile function.

## Part F - Time, sequences and cohorts

Gaps and islands, sessionization and cohort retention all reduce to "compare each row with the previous row in order",
which is `LAG` plus a cumulative `SUM`.

### Q19. Longest consecutive-day streak per user

For each user in `events`, find every run of consecutive calendar days with at least one event.

**Expected result:** user 1 has a three-day run and a two-day run; user 2 has a two-day run; users 3 and 4 have one day each.

In [ ]:
q = """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ),
    grouped AS (
        SELECT user_id, d,
               date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
        FROM daily
    )
    SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS streak_len
    FROM grouped
    GROUP BY user_id, island
    ORDER BY user_id, start_day
    """
spark.sql(q).show()
assert rows(q) == [
        (1, date(2023, 1, 1), date(2023, 1, 3), 3), (1, date(2023, 1, 5), date(2023, 1, 6), 2),
        (2, date(2023, 1, 1), date(2023, 1, 2), 2), (2, date(2023, 1, 4), date(2023, 1, 4), 1),
        (3, date(2023, 2, 1), date(2023, 2, 1), 1), (4, date(2023, 2, 1), date(2023, 2, 1), 1),
    ]
q = """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ),
    grouped AS (
        SELECT user_id, d,
               date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
        FROM daily
    )
    SELECT user_id, MAX(streak_len) AS longest FROM (
        SELECT user_id, island, COUNT(*) AS streak_len FROM grouped GROUP BY user_id, island
    )
    GROUP BY user_id
    ORDER BY user_id
    """
spark.sql(q).show()
assert rows(q) == [(1, 3), (2, 2), (3, 1), (4, 1)]

**Approach.** Deduplicate to one row per day first. Within a run, the date minus the row number is constant, so grouping by it
finds each island. `date_sub` needs an integer, so the row number is cast.

### Q20. Sessions: a new session after 30 minutes of inactivity

**Expected result:** user 3 has two sessions (the 65-minute gap starts the second); user 4 has one. Session counts per user are
5, 3, 2 and 1.

In [ ]:
q = """
    WITH ordered AS (
        SELECT event_id, user_id, event_ts,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ),
    flagged AS (
        SELECT event_id, user_id, event_ts,
               CASE WHEN prev_ts IS NULL
                      OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
                    THEN 1 ELSE 0 END AS is_new_session
        FROM ordered
    ),
    numbered AS (
        SELECT user_id,
               SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_id
        FROM flagged
    )
    SELECT user_id, MAX(session_id) AS sessions FROM numbered GROUP BY user_id ORDER BY user_id
    """
spark.sql(q).show()
assert rows(q) == [(1, 5), (2, 3), (3, 2), (4, 1)]

**Approach.** Flag the boundaries, then take a running sum of the flags. The first event per user has no previous
timestamp and must start a session explicitly. Tie-break on `event_id` so the order is deterministic.

### Q21. Cohort retention by month

Group customers by the month of their first order, then count how many were active in each later month.

**Expected result:** the January cohort has four customers and retains 4, 3, 3. The February cohort has two and retains 2, 2.

In [ ]:
q = """
    WITH first_order AS (
        SELECT customer_id, MIN(date_format(order_date, 'yyyy-MM')) AS cohort_month
        FROM orders GROUP BY customer_id
    ),
    activity AS (
        SELECT DISTINCT customer_id, date_format(order_date, 'yyyy-MM') AS activity_month FROM orders
    )
    SELECT f.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id) AS active_customers
    FROM first_order f
    JOIN activity a ON a.customer_id = f.customer_id
    GROUP BY f.cohort_month, a.activity_month
    ORDER BY f.cohort_month, a.activity_month
    """
spark.sql(q).show()
assert rows(q) == [
        ("2023-01", "2023-01", 4), ("2023-01", "2023-02", 3), ("2023-01", "2023-03", 3),
        ("2023-02", "2023-02", 2), ("2023-02", "2023-03", 2),
    ]
q = """
    WITH first_order AS (
        SELECT customer_id, MIN(date_format(order_date, 'yyyy-MM')) AS cohort_month
        FROM orders GROUP BY customer_id
    )
    SELECT cohort_month, COUNT(*) AS cohort_size FROM first_order GROUP BY cohort_month ORDER BY cohort_month
    """
spark.sql(q).show()
assert rows(q) == [("2023-01", 4), ("2023-02", 2)]

**Approach.** Three steps: each customer's first period, the distinct active (customer, period) pairs, and the join.
`COUNT(DISTINCT customer_id)` stops a customer with several orders in one month from inflating retention.

### Q22. Cumulative distinct customers by date

For each order date, show how many distinct customers had ordered on or before that date.

**Expected result:** the count reaches 6 on 2023-02-18 and stays there.

In [ ]:
q = """
    SELECT o.order_date,
           (SELECT COUNT(DISTINCT o2.customer_id) FROM orders o2 WHERE o2.order_date <= o.order_date) AS cum_customers
    FROM (SELECT DISTINCT order_date FROM orders) o
    ORDER BY o.order_date
    """
spark.sql(q).show()
assert rows(q) == [
        (date(2023, 1, 2), 1), (date(2023, 1, 5), 2), (date(2023, 1, 20), 3), (date(2023, 1, 25), 4),
        (date(2023, 2, 5), 4), (date(2023, 2, 10), 4), (date(2023, 2, 14), 5), (date(2023, 2, 18), 6),
        (date(2023, 2, 28), 6), (date(2023, 3, 5), 6), (date(2023, 3, 15), 6), (date(2023, 3, 20), 6),
        (date(2023, 3, 25), 6), (date(2023, 3, 30), 6),
    ]
q = """
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_date FROM orders GROUP BY customer_id
    )
    SELECT first_date, SUM(COUNT(*)) OVER (ORDER BY first_date) AS cum_customers
    FROM first_order
    GROUP BY first_date
    ORDER BY first_date
    """
spark.sql(q).show()
assert rows(q) == [
        (date(2023, 1, 2), 1), (date(2023, 1, 5), 2), (date(2023, 1, 20), 3),
        (date(2023, 1, 25), 4), (date(2023, 2, 14), 5), (date(2023, 2, 18), 6),
    ]

**Approach.** The correlated count recounts history for every date. The scalable version counts each customer once, at their
first order, and takes a running sum. Summing `COUNT(DISTINCT)` over raw periods double-counts customers who order in several.

## Part G - Concept questions

Answer each out loud before you read the answer. These cover indexing, isolation, schema design and query tuning.

### T1. What is an index, and what does a B-tree index actually do?

*Answer it out loud before reading on.*

An index is a separate, ordered data structure that maps key values to the physical location of rows. OLTP
engines default to a B-tree: a balanced, high-fanout tree that keeps keys sorted, so a lookup is O(log n) page
reads instead of a full table scan. Leaves are linked, which also makes range scans (`BETWEEN`, `>`,
`ORDER BY`) cheap.

The costs matter as much as the benefits: every index adds space and makes `INSERT`/`UPDATE`/`DELETE` slower
because the index must be maintained. Index the columns you filter and join on most often, and prefer composite
indexes whose column order matches the query's leading predicates (leftmost-prefix rule). An index on
`(dept_id, salary)` can serve `WHERE dept_id = ?` and `WHERE dept_id = ? ORDER BY salary`, but not
`WHERE salary = ?` alone. One footnote: DuckDB is columnar - it answers analytics directly from the columns
and keeps ART (trie) indexes only for point lookups, so the honest line is "B-trees in OLTP, almost none in a
columnar warehouse".

### T2. How do you read `EXPLAIN` / `EXPLAIN ANALYZE`, and what do you look for?

*Answer it out loud before reading on.*

`EXPLAIN` shows the physical plan the optimiser chose for a statement; add `ANALYZE` to run the query for real
and attach runtime numbers. The things to look for:

- `SEQ_SCAN`: the columnar sequential scan reads every row group. Normal for analytics and for small tables.
- `HASH_JOIN`: an equi-join built on a hash table.
- `HASH_GROUP_BY` / `PERFECT_HASH_GROUP_BY`: grouping - the perfect hash variant appears when the number of
  distinct keys is small.
- `TOP_N`: `ORDER BY ... LIMIT n` answered in a single pass instead of a full sort.
- `FILTER`: a post-scan predicate.

The plan is a *plan*, not a measurement: `EXPLAIN ANALYZE` (Part 7, skill 5) is what you measure with.

**Demo 1 - the plan for the duplicate-email query.** An unindexed `GROUP BY email` has no choice but a
sequential scan feeding a hash group-by, then a `FILTER` for the `HAVING`.

plan_text = run("EXPLAIN " + """
    SELECT email, COUNT(*) FROM employees GROUP BY email HAVING COUNT(*) > 1
""")[0][1]
print(plan_text)
ops = plan_text.upper().replace(" ", "_")
assert "SEQ_SCAN" in ops and "GROUP_BY" in ops

**Demo 2 - a filter with an index in place.** DuckDB keeps indexes as ART (trie) indexes for point lookups;
as a column store it usually answers analytics straight from the columns, which is why the plan below is
unremarkable - and why result checks, not plan shape, prove the index does not break anything.

run("CREATE INDEX IF NOT EXISTS idx_employees_dept ON employees(dept_id, salary)")
show("SELECT name, salary FROM employees WHERE dept_id = 2 ORDER BY salary DESC")
assert run("SELECT name, salary FROM employees WHERE dept_id = 2 ORDER BY salary DESC") == [
    ("Frank", 130000),
    ("Grace", 98000),
    ("Heidi", 87000),
    ("Ivan", 87000),
]
print("indexed lookup results unchanged")

**Demo 3 - confirm the index exists.** `duckdb_indexes()` is the catalogue of every created index, which is
how you check what the setup actually built.

assert run(
    "SELECT index_name FROM duckdb_indexes() WHERE index_name = 'idx_employees_dept'"
) == [("idx_employees_dept",)]
print("index registered")

### T3. What is normalisation? Explain 1NF, 2NF and 3NF, and when you would denormalise.

*Answer it out loud before reading on.*

Normalisation organises columns and tables to remove redundancy and update anomalies.

- **1NF**: every column holds a single atomic value (no repeating groups or comma-separated lists) and rows are
  unique.
- **2NF**: 1NF plus no partial dependency - every non-key column depends on the *whole* primary key, which only
  matters for composite keys.
- **3NF**: 2NF plus no transitive dependency - non-key columns depend on the key, not on another non-key column
  (for example storing `dept_name` next to `emp_id` when `dept_id` already determines it).

**BCNF** is a stricter 3NF for tables with overlapping candidate keys. You denormalise deliberately for read
performance or simplicity: star schemas keep a dimension's attributes together to avoid joins, and analytics
tables pre-join facts with dimensions. The trade-off is redundant storage and the risk of inconsistent updates,
which is why denormalised layers are usually rebuilt from a normalised source of truth.

**Follow-up.** Where does a star schema sit relative to 3NF? (Answer: a dimension is typically 2NF/3NF-ish, and
the fact table is intentionally denormalised.)

### T4. What does ACID mean, and what are the isolation levels?

*Answer it out loud before reading on.*

- **Atomicity**: all statements in a transaction commit or none do.
- **Consistency**: a transaction moves the database from one valid state to another (constraints hold).
- **Isolation**: concurrent transactions do not see each other's uncommitted work.
- **Durability**: once committed, data survives a crash.

The SQL standard defines four isolation levels, each permitting fewer anomalies:

| level | dirty read | non-repeatable read | phantom read |
| --- | --- | --- | --- |
| Read uncommitted | yes | yes | yes |
| Read committed | no | yes | yes |
| Repeatable read | no | no | yes |
| Serializable | no | no | no |

Most engines default to Read Committed (Postgres, Oracle) or Repeatable Read (MySQL InnoDB); embedded engines
serialise instead - DuckDB allows one writer transaction per database at a time. In practice you also meet
**snapshot isolation** and **write skew** (MVCC): Postgres' Repeatable Read is a snapshot isolation level that
prevents write skew by aborting conflicting transactions. Isolation is a correctness-versus-throughput dial:
stronger isolation means more locking or more aborts.

### T5. `GROUP BY` vs window functions: what is the difference?

*Answer it out loud before reading on.*

`GROUP BY` collapses rows: the result has one row per group, so you lose the individual rows. Window functions
keep every row and compute the aggregate (or ranking) across a defined window, which is why they can add
"department average" or "running total" as extra columns.

```sql
-- one row per department
SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id;

-- every employee keeps their row and gains the department average
SELECT name, salary, AVG(salary) OVER (PARTITION BY dept_id) FROM employees;
```

Windows are evaluated after `WHERE`, `GROUP BY` and `HAVING`, which is why you cannot filter on a window alias
in `WHERE` - use an outer query, or `QUALIFY`, which DuckDB, Snowflake and BigQuery support directly (Part 7,
skill 1). Window functions can also rank (`ROW_NUMBER`/`RANK`/`DENSE_RANK`) and access other rows
(`LAG`/`LEAD`, `FIRST_VALUE`), which `GROUP BY` cannot do without a self-join.

### T6. `UNION` vs `UNION ALL`

*Answer it out loud before reading on.*

`UNION` concatenates two result sets and removes duplicates, which requires a sort or hash and is therefore slower
and memory-hungry. `UNION ALL` simply concatenates, keeping duplicates. The rule of thumb: use `UNION ALL` unless
you specifically need deduplication, which is almost never in ETL where duplicates usually signal a bug rather
than something to hide. Both require compatible column counts and types; only the first query's column names are
used. `INTERSECT` and `EXCEPT` (minus) are the set-difference counterparts.

### T7. Explain `NULL` semantics, and the traps around `NOT IN` and `COUNT`.

*Answer it out loud before reading on.*

`NULL` means "unknown", not "empty". Any comparison with `NULL` yields `UNKNOWN`, and `WHERE` keeps only rows
where the predicate is `TRUE`. So `salary = NULL` and `salary <> NULL` both return nothing; use `IS NULL` /
`IS NOT NULL`. Three-valued logic propagates through `AND`/`OR`: `TRUE OR NULL` is `TRUE`, but `TRUE AND NULL` is
`NULL`.

Traps:

- `NOT IN (subquery)` returns **no rows** if the subquery yields even one `NULL`, because `x <> NULL` is unknown.
  Prefer `NOT EXISTS`.
- `COUNT(*)` counts rows; `COUNT(col)` counts non-null values; `COUNT(DISTINCT col)` ignores nulls too.
- `SUM`/`AVG`/`MIN`/`MAX` skip nulls, so `AVG` over a column with nulls divides by the non-null count. `SUM` of
  no rows is `NULL`, not 0 - wrap it in `COALESCE`.
- `NULL = NULL` is unknown, so joins on nullable keys behave surprisingly.
- Many engines index nulls (Oracle does not by default), and `GROUP BY` treats all nulls as one group.

### T8. `HAVING` vs `WHERE`

*Answer it out loud before reading on.*

`WHERE` filters individual rows *before* grouping and aggregation; `HAVING` filters groups *after* aggregation.
`WHERE status = 'completed'` removes rows, `HAVING COUNT(*) > 1` removes whole groups. Because of that order,
`HAVING` may reference aggregates and `WHERE` may not; conversely, filtering in `WHERE` is usually cheaper
because fewer rows reach the aggregation step. A common interview follow-up is "can you use an aggregate in
`WHERE`?" - no, and the fix is either `HAVING` or a subquery/CTE that computes the aggregate first.

### T9. CTE vs subquery: when do you use each?

*Answer it out loud before reading on.*

A CTE (`WITH name AS (...)`) is a named subquery. It improves readability, lets you reference the same result
more than once, and can be recursive. Whether it is materialised or inlined depends on the engine: Postgres 12+
inlines non-recursive CTEs unless marked `MATERIALIZED`; DuckDB, BigQuery and Snowflake treat them as named
subqueries the optimiser may inline. Historically, CTEs were an "optimisation fence" in older Postgres, which is
why the old advice was "CTEs are slow" - measure your engine rather than assume. Use a CTE for clarity or
reuse; use a plain subquery for a one-off inline filter. For hierarchies (org charts, bill of materials, graph
traversal), a recursive CTE is the tool:

```sql
WITH RECURSIVE subordinates AS (
    SELECT emp_id, manager_id FROM employees WHERE emp_id = 1
    UNION ALL
    SELECT e.emp_id, e.manager_id
    FROM employees e JOIN subordinates s ON e.manager_id = s.emp_id
)
SELECT * FROM subordinates;
```

### T10. Star vs snowflake schema

*Answer it out loud before reading on.*

A **star schema** has one central fact table (one row per business event, with foreign keys and measures) joined
directly to denormalised dimension tables. It is simple, requires few joins, and is what most BI tools assume. A
**snowflake schema** normalises dimensions into sub-dimensions (product -> category -> department), saving
storage and removing redundancy but requiring more joins.

In practice, star wins for analytics because storage is cheap and joins cost time; snowflaking is worth it when a
dimension is enormous or shared across many marts. The related idea is **one big table (OBT)** / wide denormalised
tables, which push the join cost to the transformation layer. Fact-table design also distinguishes
transaction facts (one row per event), periodic snapshots (one row per entity per period) and accumulating
snapshots (one row per process instance, updated as it moves through stages).

### T11. What is a slowly changing dimension (SCD), and how does Type 2 work?

*Answer it out loud before reading on.*

An SCD is a dimension attribute that changes over time rather than staying fixed. The types:

- **Type 0**: never changes (a true natural key).
- **Type 1**: overwrite in place - no history, simplest, loses the old value.
- **Type 2**: add a new row per version, with `valid_from`/`valid_to` (or `start_date`/`end_date`) and often a
  `is_current` flag and a new surrogate key. History is preserved and facts can be joined to the version that
  was current when the event happened.
- **Type 3**: keep a limited number of previous values as extra columns (`current_region`, `prior_region`).
- **Type 6**: a hybrid of 1, 2 and 3.

Type 2 is the standard interview answer. The implementation closes the current row (`valid_to = today`,
`is_current = false`) and inserts a new row with `valid_from = today`. Getting the boundaries right matters:
`valid_to` is usually exclusive (`[valid_from, valid_to)`), and a common bug is overlapping or gapped ranges.
Loading Type 2 with a merge/incremental model (`dbt snapshot` is the classic tool) is the day-to-day reality.

### T12. What is partitioning, and how does it help?

*Answer it out loud before reading on.*

Partitioning physically splits a large table into smaller pieces by a key - usually a date range, or a hash or
list of keys - while queries still see one logical table. The main win is **partition pruning**: a query filtered
on the partition key reads only the relevant partitions instead of the whole table. It also enables partition-wise
maintenance (drop an old partition instead of `DELETE`ing rows) and parallel scans.

Things to know: the partition key should appear in the `WHERE` clause of the important queries, or pruning cannot
happen; too many small partitions hurt (metadata overhead); and partitioning is not the same as clustering or
bucketing, which order rows *within* a partition. Many warehouses combine both: partition by day, cluster by
customer id.

### T13. How do you approach query optimisation?

*Answer it out loud before reading on.*

A repeatable checklist beats guessing:

1. **Measure first.** Get the actual plan and timings, not a hunch.
2. **Read the plan** (`EXPLAIN` / `EXPLAIN ANALYZE`; Part 7, skill 5 walks a DuckDB plan operator by operator -
   `SEQ_SCAN`, `HASH_JOIN`, `HASH_GROUP_BY`, `TOP_N`). Look for full scans on big tables, bad join orders, and
   spills/sorts.
3. **Reduce rows early.** Push `WHERE` predicates down, filter before joining, and select only the columns you
   need - `SELECT *` blocks covering indexes and moves data you will never read.
4. **Index for the access pattern** you actually measured: composite indexes matching the leading predicates and
   the `ORDER BY`; avoid indexing low-cardinality columns on their own.
5. **Fix the join strategy** where needed: statistics, correct data types, and explicit join keys.
6. **Pre-aggregate or materialise** expensive repeated work; incremental models instead of full rebuilds.
7. **Partition and prune** large tables.
8. **Verify with the new plan and timings**, and watch the write cost you added.

Common anti-patterns: functions on the indexed column (`WHERE date(created_at) = ...`), implicit type casts,
`SELECT DISTINCT` to paper over a bad join, and `NOT IN` with nullable subqueries.

### T14. Clustered vs non-clustered indexes, and what is a covering index?

*Answer it out loud before reading on.*

A **clustered** index defines the physical order of the table's rows - there can be only one, and in InnoDB and
SQL Server the primary key is clustered by default. Because rows are stored in key order, range scans on the
clustered key are very fast, and the index *is* the data (no extra lookup). A **non-clustered** index is a
separate structure: it stores the key plus a pointer (row id / primary key) back to the row, so a lookup may need
a second read, called a **key lookup** or **bookmark lookup**.

A **covering index** contains all the columns a query needs, so the engine can answer it from the index alone -
this is the "index-only scan". For example, an index on `(dept_id, salary)` covers
`SELECT salary FROM employees WHERE dept_id = ?` without touching the table. In Postgres you make this explicit
with `INCLUDE` columns, and in InnoDB the primary key is implicitly appended to every secondary index, which is
why keeping the primary key narrow matters. DuckDB is the outlier: as a column store its scans already read only
the referenced columns, and its indexes are ART tries for point lookups rather than covering structure.
Trade-offs are the same as any index: extra storage and slower writes.

## Where to go next

Re-do the problems you got wrong, and the [teaching notebooks](../03_sql_interview_questions.ipynb) for the topics behind them.